In [29]:
import traci
import numpy as np

USE_GUI = True
sumo_binary = "sumo-gui" if USE_GUI else "sumo"
sumo_cmd = [
    sumo_binary,
    "-c",
    "intersection.sumocfg",
    "--start",
    "--quit-on-end",
]

# Close stale connection if a previous run crashed.
try:
    traci.close()
except Exception:
    pass

waiting_times = []
step = 0
total_cumulative_wait = 0.0

try:
    traci.start(sumo_cmd)
    traci.trafficlight.setProgram("C", "movement_control")

    while traci.simulation.getMinExpectedNumber() > 0:
        traci.simulationStep()
        total_wait = sum(
            traci.lane.getWaitingTime(lane_id)
            for lane_id in traci.lane.getIDList()
        )
        waiting_times.append(total_wait)
        total_cumulative_wait += total_wait
        step += 1
except Exception as e:
    print("Simulation ended with error:", e)
finally:
    try:
        traci.close()
    except Exception:
        pass

# Enhanced metrics reporting
print("\n" + "="*60)
print("✓ BASELINE FIXED-TIME SIGNAL CONTROL EVALUATION")
print("="*60)

print(f"\n📊 Simulation Metrics:")
print(f"  Total Simulation Steps: {step}")

if waiting_times:
    avg_wait = sum(waiting_times) / len(waiting_times)
    max_wait = max(waiting_times)
    min_wait = min(waiting_times)
    
    print(f"\n⏱️  Traffic Wait Time Analysis:")
    print(f"  Average Wait Time (per step): {round(avg_wait, 2)} seconds")
    print(f"  Maximum Wait Time: {round(max_wait, 2)} seconds")
    print(f"  Minimum Wait Time: {round(min_wait, 2)} seconds")
    print(f"  Total Cumulative Wait: {round(total_cumulative_wait, 2)} seconds")
    print(f"  Average Wait per Step: {round(total_cumulative_wait / max(step, 1), 2)} seconds")
else:
    print(f"\n⏱️  Traffic Wait Time Analysis:")
    print(f"  No waiting time recorded")


✓ BASELINE FIXED-TIME SIGNAL CONTROL EVALUATION

📊 Simulation Metrics:
  Total Simulation Steps: 2658

⏱️  Traffic Wait Time Analysis:
  Average Wait Time (per step): 1181.11 seconds
  Maximum Wait Time: 4862.0 seconds
  Minimum Wait Time: 0.0 seconds
  Total Cumulative Wait: 3139392.0 seconds
  Average Wait per Step: 1181.11 seconds
